# Ray Workflows

This notebook demonstrates ray tracing and ray-based integration on a spherical BATSRUS sample (SC).

You will walk through:
- dataset loading
- octree/interpolator setup
- single-ray diagnostics
- piecewise-linear ray validation
- 2D camera-style ray integrals


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from starwinds_readplt.dataset import Dataset

from batcamp import Octree
from batcamp import OctreeInterpolator
from batcamp import OctreeRayInterpolator
from batcamp import OctreeRayTracer

from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from batcamp.ray import HEX_TETS_INDEX, TET_FACES_INDEX

import pooch


### 1. Load The Spherical Sample

Download and open the SC sample file (3d__var_4_n00044000.plt) from the Zenodo archive using pooch. This file is spherical (tree_coord="rpa").


In [ ]:
path = pooch.retrieve(
    url='https://zenodo.org/records/7110555/files/run-Sun-G2211.tar.gz',
    known_hash='c31a32aab08cc20d5b643bba734fd7220e6b369e691f55f88a3a08cc5b2a2136',
    progressbar=False,
    processor=pooch.Untar(members=['run-Sun-G2211/SC/IO2/3d__var_4_n00044000.plt']),
)[0]
path

### 2. Build Dataset, Tree, And Interpolator

Create a spherical octree and bind an OctreeInterpolator for density (Rho [g/cm^3]). The printed tree/interpolator summaries are a quick sanity check before ray work.


In [ ]:
ds = Dataset.from_file(path)
print(ds)

interp = OctreeInterpolator(ds, ['Rho [g/cm^3]'])
print(interp)


### 3. 3D Traversal Visualization

Build one diagnostic ray and render traversed hexahedral cells in 3D with per-segment coloring.


In [ ]:
# Build one diagnostic ray for visualization.
r_shell = 4.0
p_start = r_shell * np.array([1.0, 0.0, 0.0], dtype=float)
p_end = r_shell * np.array([-0.25, 0.96, 0.11], dtype=float)
p_end = r_shell * (p_end / np.linalg.norm(p_end))
origin = p_start
direction = p_end - p_start
t0, t1 = 0.0, float(np.linalg.norm(direction))
segments = OctreeRayTracer(interp.tree).trace(origin, direction, t0, t1)

if not segments:
    raise RuntimeError('Ray did not traverse any octree cells; adjust ray setup.')
ray_dir = direction / np.linalg.norm(direction)
n_seg = len(segments)
cmap = plt.colormaps['turbo']
colors = cmap(np.linspace(0.0, 1.0, max(n_seg, 2)))
def cube_index(r_bit, polar_bit, azimuth_bit):
    return int(r_bit + 2 * polar_bit + 4 * azimuth_bit)
cube_faces = [
    [cube_index(0, 0, 0), cube_index(0, 1, 0), cube_index(0, 1, 1), cube_index(0, 0, 1)],
    [cube_index(1, 0, 0), cube_index(1, 0, 1), cube_index(1, 1, 1), cube_index(1, 1, 0)],
    [cube_index(0, 0, 0), cube_index(0, 0, 1), cube_index(1, 0, 1), cube_index(1, 0, 0)],
    [cube_index(0, 1, 0), cube_index(1, 1, 0), cube_index(1, 1, 1), cube_index(0, 1, 1)],
    [cube_index(0, 0, 0), cube_index(1, 0, 0), cube_index(1, 1, 0), cube_index(0, 1, 0)],
    [cube_index(0, 0, 1), cube_index(0, 1, 1), cube_index(1, 1, 1), cube_index(1, 0, 1)],
]
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
all_faces = []
all_facecolors = []
extent_points = []
for idx, seg in enumerate(segments):
    color = colors[idx]
    cid = int(seg.cell_id)
    p0 = origin + float(seg.t_enter) * ray_dir
    p1 = origin + float(seg.t_exit) * ray_dir
    pm = 0.5 * (p0 + p1)
    ax.plot([p0[0], p1[0]], [p0[1], p1[1]], [p0[2], p1[2]], color=color, linewidth=3.0)
    ax.text(pm[0], pm[1], pm[2], str(idx), color=color, fontsize=7)
    corner_ids = interp._corners[cid]
    cell_xyz_raw = interp.lookup.points[corner_ids]
    corner_order = interp._bin_to_corner[cid]
    cell_xyz = cell_xyz_raw[corner_order]
    for face in cube_faces:
        all_faces.append(cell_xyz[np.array(face, dtype=np.int64)])
        all_facecolors.append(color)
    extent_points.extend([p0, p1])
    extent_points.extend(cell_xyz)
poly = Poly3DCollection(
    all_faces,
    facecolors=all_facecolors,
    # edgecolors=all_facecolors,
    linewidths=0.4,
    alpha=0.12,
)
ax.add_collection3d(poly)
extent = np.array(extent_points, dtype=float)
mins = np.min(extent, axis=0)
maxs = np.max(extent, axis=0)
center = 0.5 * (mins + maxs)
radius = 0.5 * float(np.max(maxs - mins))
ax.set_xlim(center[0] - radius, center[0] + radius)
ax.set_ylim(center[1] - radius, center[1] + radius)
ax.set_zlim(center[2] - radius, center[2] + radius)
ax.set_xlabel('X [R]')
ax.set_ylabel('Y [R]')
ax.set_zlabel('Z [R]')
ax.set_title('Ray traversal with hexahedral cell surfaces (r=4 to r=4 endpoints)')
norm = plt.Normalize(vmin=0, vmax=max(n_seg - 1, 1))
sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.04)
cbar.set_label('ray segment index (0 = first cell)')
plt.tight_layout()
plt.show()


### 4. Camera-Style Column Image Along +X

Launch a grid of rays and integrate rho^2 along each ray using adaptive midpoint quadrature (adaptive_midpoint_rule).

Output: ray_rho2_x_view.png and an on-screen image in log10(Integral rho^2 ds).


In [ ]:
# Camera-style projection from -x toward +x: integrate rho^2 along x-directed rays.
# Ray-first design: launch from an interior plane and integrate with bulk adaptive midpoint rays.
ray_img = OctreeRayInterpolator(interp)


def integrate_rho2_with_rays(ray_obj, origins_xyz, direction_xyz, t_end, *, chunk_size=4096):
    out = np.full(origins_xyz.shape[0], np.nan, dtype=float)

    mids, weights, offsets = ray_obj.adaptive_midpoint_rule(
        origins_xyz,
        direction_xyz,
        0.0,
        float(t_end),
        chunk_size=chunk_size,
    )
    if mids.shape[0] == 0:
        return out

    rho_mid = np.asarray(
        interp(mids, query_coord='xyz', log_outside_domain=False),
        dtype=float,
    ).reshape(-1)

    vals = np.full(origins_xyz.shape[0], np.nan, dtype=float)
    for i in range(vals.shape[0]):
        s = int(offsets[i])
        e = int(offsets[i + 1])
        if e <= s:
            continue
        seg_rho = rho_mid[s:e]
        seg_w = weights[s:e]
        finite = np.isfinite(seg_rho)
        if not np.any(finite):
            continue
        vals[i] = np.sum((seg_rho[finite] * seg_rho[finite]) * seg_w[finite])
    return vals


dmin, dmax = interp.tree.domain_bounds(coord='xyz')
xmin, xmax = float(dmin[0]), float(dmax[0])
ymin, ymax = float(dmin[1]), float(dmax[1])
zmin, zmax = float(dmin[2]), float(dmax[2])

ny, nz = 32, 32
# Increase ny/nz for higher-resolution images.
y_grid = np.linspace(ymin, ymax, ny)
z_grid = np.linspace(zmin, zmax, nz)
Yg, Zg = np.meshgrid(y_grid, z_grid, indexing='xy')

x0 = xmin + 1e-6 * (xmax - xmin)
origins = np.column_stack(
    (
        np.full(Yg.size, x0, dtype=float),
        Yg.ravel(),
        Zg.ravel(),
    )
)
direction = np.array([1.0, 0.0, 0.0], dtype=float)

rho2_column = integrate_rho2_with_rays(ray_img, origins, direction, (xmax - xmin) * 0.999999)
rho2_image = rho2_column.reshape(Yg.shape)
plot_data = np.where(
    np.isfinite(rho2_image) & (rho2_image > 0.0),
    np.log10(rho2_image),
    np.nan,
)

valid = np.isfinite(plot_data)
if np.any(valid):
    vmin = float(np.nanpercentile(plot_data[valid], 2.0))
    vmax = float(np.nanpercentile(plot_data[valid], 98.0))
else:
    vmin, vmax = -1.0, 1.0
cmap = plt.colormaps['magma'].copy()
cmap.set_bad('black')
fig, ax = plt.subplots(figsize=(7.5, 6.5))
ax.set_facecolor('black')
im = ax.imshow(
    plot_data,
    origin='lower',
    extent=[ymin, ymax, zmin, zmax],
    aspect='equal',
    cmap=cmap,
    vmin=vmin,
    vmax=vmax,
)
ax.set_xlabel('y')
ax.set_ylabel('z')
ax.set_title('View from -x to +x: log10(Integral rho^2 ds)')
cb = fig.colorbar(im, ax=ax)
cb.set_label('log10(Integral rho^2 ds)')
out_png = 'ray_rho2_x_view.png'
fig.savefig(out_png, dpi=180, bbox_inches='tight')
plt.show()


### 5. Rotated Camera Plane

Rotate the camera/ray direction around z and repeat the same rho^2 column integration to produce a second projection.

Output: ray_rho2_rotated_view.png.


In [ ]:
# Rotate the image plane around z and integrate rho^2 along the rotated rays.
dmin, dmax = interp.tree.domain_bounds(coord='xyz')

xmin, xmax = float(dmin[0]), float(dmax[0])
ymin, ymax = float(dmin[1]), float(dmax[1])
zmin, zmax = float(dmin[2]), float(dmax[2])

corners = np.array(
    [
        [x, y, z]
        for x in (xmin, xmax)
        for y in (ymin, ymax)
        for z in (zmin, zmax)
    ],
    dtype=float,
)

angle_deg = 20.0
angle = np.deg2rad(angle_deg)
ray_dir = np.array([np.cos(angle), np.sin(angle), 0.0], dtype=float)
ray_dir = ray_dir / np.linalg.norm(ray_dir)

e_u = np.array([-ray_dir[1], ray_dir[0], 0.0], dtype=float)
e_v = np.array([0.0, 0.0, 1.0], dtype=float)

u_c = corners @ e_u
v_c = corners @ e_v
w_c = corners @ ray_dir
u_min, u_max = float(np.min(u_c)), float(np.max(u_c))
v_min, v_max = float(np.min(v_c)), float(np.max(v_c))
w_min, w_max = float(np.min(w_c)), float(np.max(w_c))

nu, nv = 32, 32
# Increase nu/nv for higher-resolution images.
u_grid = np.linspace(u_min, u_max, nu)
v_grid = np.linspace(v_min, v_max, nv)
Ug, Vg = np.meshgrid(u_grid, v_grid, indexing='xy')

w0 = w_min + 1e-6 * (w_max - w_min)
origins_rot = (
    Ug.reshape(-1, 1) * e_u[None, :]
    + Vg.reshape(-1, 1) * e_v[None, :]
    + w0 * ray_dir[None, :]
)

rho2_rot = integrate_rho2_with_rays(
    ray_img,
    origins_rot,
    ray_dir,
    (w_max - w_min) * 0.999999,
)

rho2_rot_img = rho2_rot.reshape(Ug.shape)
plot_data = np.where(
    np.isfinite(rho2_rot_img) & (rho2_rot_img > 0.0),
    np.log10(rho2_rot_img),
    np.nan,
)

valid = np.isfinite(plot_data)
if np.any(valid):
    vmin = float(np.nanpercentile(plot_data[valid], 2.0))
    vmax = float(np.nanpercentile(plot_data[valid], 98.0))
else:
    vmin, vmax = -1.0, 1.0
cmap = plt.colormaps['magma'].copy()
cmap.set_bad('black')
fig, ax = plt.subplots(figsize=(7.5, 6.5))
ax.set_facecolor('black')
im = ax.imshow(
    plot_data,
    origin='lower',
    extent=[u_min, u_max, v_min, v_max],
    aspect='equal',
    cmap=cmap,
    vmin=vmin,
    vmax=vmax,
)
ax.set_xlabel('u (rotated around z)')
ax.set_ylabel('z')
ax.set_title(f'View rotated {angle_deg:.0f} deg around z: log10(Integral rho^2 ds)')
cb = fig.colorbar(im, ax=ax)
cb.set_label('log10(Integral rho^2 ds)')
out_png = 'ray_rho2_rotated_view.png'
fig.savefig(out_png, dpi=180, bbox_inches='tight')
plt.show()
